# SemMap · Гипотеза H1: nSBM, отложенные связи и нулевые графы — Colab 12 CPU + GPU

**Назначение:** воспроизводимый эксперимент на ConceptNet по сравнению обычной SBM, модели с поправкой на степени (DC-SBM) и вложенной DC-SBM, с повторными инициализациями и двумя ансамблями контрольных графов.

**Важное разделение вычислений:** построение вложенных SBM и проверка H1 работают на **CPU/OpenMP** с ограничением максимум 12 доступных ядер; **GPU (CUDA)** используется для пакетного вычисления эмпирических блочных ядер W после обнаружения иерархии. Ускорение GPU **не относится** к оцениванию nSBM и не доказывает существование графонного/графексного предела. S и I здесь не оцениваются.

**Профили:** SMOKE — искусственный граф, без нулевых контролей; PILOT — ConceptNet с небольшим числом контролей; FULL — ConceptNet с 10 перезапусками и 99 контролями *каждого* из двух ансамблей (итого 5970 подгонок моделей). SMOKE и PILOT не позволяют делать научный вывод об H1.

**Хранение:** код закреплён на Git SHA; вычисления в /content; выборка и контрольные точки на Google Drive, автоматический возврат в режим RESUME после обрыва Colab. Для FULL подготовьте исходный пятиколоночный TSV/GZ с идентификаторами утверждений; нельзя заменять его агрегированными selected_edges.jsonl.

Настройте аппаратный ускоритель Colab в меню **Среда выполнения → Сменить среду выполнения → GPU**. Доступность 12 CPU и GPU определяется по фактическим ресурсам сеанса.

In [ ]:
# ====== ЕДИНАЯ КОНФИГУРАЦИЯ: меняйте только эту ячейку ======
REPOSITORY_URL = "https://github.com/SemanticMap/semgraphex.git"
SOURCE_REVISION = "40816f53288c5527b64582a608af0c4924646c96"
REPO_DIR = "/content/semmap_h1_repo"

RUN_PROFILE = "SMOKE"                 # SMOKE | PILOT | FULL
RUN_ID = "h1-synthetic-smoke-01"       # новое имя для других данных/параметров!
SEED = 42
CPU_BUDGET = 12                       # предел, но не обещание 12 выделенных CPU
REQUIRE_GPU = True                    # если GPU отсутствует, завершить с понятной ошибкой
GPU_BATCH_SIZE = 150_000
GPU_MAX_LEVELS = 4
CHECKPOINT_INTERVAL_SECONDS = 90

USE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/SemanticMap"
# Заполните путь к настоящему 5-колоночному ConceptNet TSV/TSV.GZ для PILOT/FULL:
CONCEPTNET_TSV_ON_DRIVE = ""           # например: /content/drive/MyDrive/.../conceptnet_en_100k.tsv
RUN_CALIBRATION = False               # отдельная очень ресурсоёмкая серия синтетических проверок
CALIBRATION_SIMULATIONS = 20
CALIBRATION_RESTARTS = 3
CALIBRATION_NULLS = 99

PROFILES = {
    "SMOKE": {"restarts": 1, "nulls": 0, "negatives": 4},
    "PILOT": {"restarts": 2, "nulls": 9, "negatives": 30},
    "FULL": {"restarts": 10, "nulls": 99, "negatives": 100},
}
assert RUN_PROFILE in PROFILES
assert 1 <= CPU_BUDGET <= 128
assert CHECKPOINT_INTERVAL_SECONDS >= 20
print("Профиль", RUN_PROFILE, PROFILES[RUN_PROFILE], "| код", SOURCE_REVISION[:12])

In [ ]:
# 1. Проверка реальных ресурсов: ядра (включая cgroup), RAM, диск, CUDA
from pathlib import Path
import os, sys, shutil, subprocess, json, time, platform, hashlib

def usable_cpus():
    count = os.cpu_count() or 1
    if hasattr(os, "sched_getaffinity"):
        count = min(count, len(os.sched_getaffinity(0)))
    cpu_max = Path("/sys/fs/cgroup/cpu.max")
    if cpu_max.exists():
        quota, period = cpu_max.read_text().split()
        if quota != "max":
            count = min(count, max(1, -(-int(quota) // int(period))))
    return max(1, count)

AVAILABLE_CPU = usable_cpus()
ALLOTTED_CPU = min(CPU_BUDGET, AVAILABLE_CPU)
FREE_DISK_GB = shutil.disk_usage("/content").free / (1024 ** 3)
ram_gb = (os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")) / 1024**3
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
GPU_VISIBLE = gpu.returncode == 0 and "GPU" in gpu.stdout
print({"python": sys.version.split()[0], "os": platform.platform(),
       "available_cpus": AVAILABLE_CPU, "requested_budget": CPU_BUDGET,
       "effective_cpu_budget": ALLOTTED_CPU,
       "free_disk_gb": round(FREE_DISK_GB, 1),
       "host_available_ram_gb": round(ram_gb, 1),
       "cuda_gpu": gpu.stdout.strip() if GPU_VISIBLE else "not detected"})
if ALLOTTED_CPU < CPU_BUDGET:
    print(f"ВНИМАНИЕ: вместо запрошенных {CPU_BUDGET} доступно {ALLOTTED_CPU}; не обещаем отсутствующие ядра.")
if FREE_DISK_GB < 7:
    raise RuntimeError("Для установки graph-tool требуется как минимум ~7 GiB свободного /content")
if REQUIRE_GPU and not GPU_VISIBLE:
    raise RuntimeError("GPU не обнаружен: выберите GPU в настройках Colab либо REQUIRE_GPU=False")

In [ ]:
# 2. Google Drive: только долговременные данные и контрольные точки
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path(DRIVE_PROJECT_ROOT)
WORK_ROOT = Path("/content/semmap_h1_work")
BIN_ROOT = Path("/content/semmap_h1_bin")
ENV_ROOT = Path("/content/semmap_h1_env")
REPOSITORY = Path(REPO_DIR)
for directory in (WORK_ROOT, BIN_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
RUN_LOCAL = WORK_ROOT / "runs" / RUN_ID
RUN_DRIVE = DRIVE_ROOT / "colab" / "hypothesis" / "runs" / RUN_ID
if USE_DRIVE:
    if not Path("/content/drive").is_mount():
        raise RuntimeError("Google Drive не смонтирован")
    RUN_DRIVE.mkdir(parents=True, exist_ok=True)
print("Локальная рабочая папка:", RUN_LOCAL)
print("Постоянное хранилище:", RUN_DRIVE if USE_DRIVE else "(отключено; файлы могут быть потеряны)")

In [ ]:
# 3. Git: получаем код из репозитория и фиксируем именно указанную ревизию
if not (REPOSITORY / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "75", "--branch",
                    "feature/hypothesis-mvp-parallel-gpu", REPOSITORY_URL,
                    str(REPOSITORY)], check=True)
else:
    subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "--depth", "75",
                    "origin", "feature/hypothesis-mvp-parallel-gpu"], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach",
                SOURCE_REVISION], check=True)
checked = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
assert checked == SOURCE_REVISION
print("Закреплён код semgraphex:", checked)

## 4. Graph-tool без несовместимости Python

Colab и системный Ubuntu Python могут иметь разные версии. Поэтому используем **изолированное окружение micromamba** с graph-tool-base 2.98 и Python 3.12. Не заменяем Python работающего блокнота; все вычисления nSBM идут отдельным процессом окружения.

Пакеты устанавливаются из conda-forge. После установки ноутбук **обязательно проверяет реальный импорт graph-tool**, а не скрывает ошибку или переключает алгоритм на другой. Пакеты не обновляются без необходимости в базовом Colab. Для повторяемости сохраняется список установленных версий.

In [ ]:
# 4. Установка micromamba и рабочего окружения graph-tool (один раз на новый runtime)
MAMBA = BIN_ROOT / "bin" / "micromamba"
ENV_PY = ENV_ROOT / "bin" / "python"
BOOTSTRAP_LOG = WORK_ROOT / "bootstrap.log"

def checked_log(command, *, cwd=None, env=None, log_path=BOOTSTRAP_LOG):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(map(str, command)) + "\n")
        log.flush()
        outcome = subprocess.run(command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
    if outcome.returncode:
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-45:]
        raise RuntimeError(f"Команда завершилась с ошибкой {outcome.returncode}:\n" + "\n".join(tail))
    return outcome

if not MAMBA.exists():
    BIN_ROOT.mkdir(parents=True, exist_ok=True)
    checked_log(["bash", "-lc", "set -o pipefail; curl -fsSL "
                 "https://micro.mamba.pm/api/micromamba/linux-64/latest "
                 "| tar -xj -C " + str(BIN_ROOT) + " bin/micromamba"])
mamba_env = dict(os.environ, MAMBA_ROOT_PREFIX="/content/semmap_mamba")
if not ENV_PY.exists():
    checked_log([str(MAMBA), "create", "-y", "-p", str(ENV_ROOT),
                 "-c", "conda-forge", "--strict-channel-priority",
                 "python=3.12", "graph-tool-base=2.98", "numpy",
                 "scipy", "scikit-learn", "pyyaml", "networkx",
                 "pytest", "psutil", "threadpoolctl"], env=mamba_env)
version = subprocess.check_output(
    [str(ENV_PY), "-c",
     "import graph_tool.all as gt; import numpy; print(gt.__version__,numpy.__version__)"],
    text=True, stderr=subprocess.STDOUT,
).strip()
print("Исполняемое окружение:", ENV_PY, "\n", version)
CONDA_SPEC = WORK_ROOT / "environment_packages.json"
CONDA_SPEC.write_text(subprocess.check_output(
    [str(MAMBA), "list", "-p", str(ENV_ROOT), "--json"],
    env=mamba_env, text=True), encoding="utf-8")
if USE_DRIVE:
    copy_target = RUN_DRIVE / "environment_packages.json"
    if copy_target.exists() and copy_target.read_bytes() != CONDA_SPEC.read_bytes():
        print("ВНИМАНИЕ: окружение отличается от предыдущего запуска; см. версии библиотек.")
    else:
        copy_if_environment = copy_target.with_suffix(".part")
        shutil.copy2(CONDA_SPEC, copy_if_environment)
        os.replace(copy_if_environment, copy_target)

In [ ]:
# 5. Реальные интеграционные тесты graph-tool (до долгого эксперимента)
PYTHONPATH = str(REPOSITORY / "src") + ":" + str(REPOSITORY)
CPU_ENV = dict(os.environ, PYTHONPATH=PYTHONPATH, OPENBLAS_NUM_THREADS="1",
               MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1", OMP_NUM_THREADS="1",
               PYTHONUNBUFFERED="1")
checked_log([str(ENV_PY), "-m", "pytest", "-q",
             "tests/test_h1_graph_tool_integration.py",
             "tests/test_h1_science.py",
             "tests/test_h1_runner.py"],
            cwd=REPOSITORY, env=CPU_ENV, log_path=WORK_ROOT / "h1_preflight_tests.log")
print("Интеграционные тесты graph-tool прошли в этом runtime.")

In [ ]:
# 6. GPU-first: CuPy только для проекции W. Ранний CUDA/CPU smoke.
import numpy as np
sys.path.insert(0, str(REPOSITORY / "src"))
from semmap_haken.hypothesis.graph import TypedGraph
from semmap_haken.hypothesis.projection import estimate_block_kernel

if GPU_VISIBLE:
    try:
        import cupy as cp
    except ImportError:
        # Colab L4 с CUDA 12.x; для иной основной версии CUDA нужен другой wheel.
        checked_log([sys.executable, "-m", "pip", "install", "--quiet", "cupy-cuda12x>=13,<15"],
                    log_path=WORK_ROOT / "cupy_install.log")
        import cupy as cp
    assert cp.cuda.runtime.getDeviceCount() > 0
    print("CUDA device:", cp.cuda.runtime.getDeviceProperties(0)["name"])
smoke_g = TypedGraph(
    ("a", "b", "c", "d"), ("/r/IsA",),
    np.array([0, 0, 1, 2]), np.array([1, 2, 2, 3]),
    np.array([0, 0, 0, 0], dtype=np.int32),
    np.ones(4), ("a1", "a2", "a3", "a4"))
smoke_p = np.array([0, 0, 1, 1], dtype=np.int64)
ref = estimate_block_kernel(smoke_g, smoke_p, device="cpu", batch_size=2)
if GPU_VISIBLE:
    accel = estimate_block_kernel(smoke_g, smoke_p, device="cuda", batch_size=2)
    assert accel["backend"] == "cuda"
    assert [(v["source_block"],v["target_block"],v["edge_count"])
            for v in accel["entries"]] == [
        (v["source_block"],v["target_block"],v["edge_count"]) for v in ref["entries"]]
    assert np.allclose([v["weight_sum"] for v in ref["entries"]],
                       [v["weight_sum"] for v in accel["entries"]])
    print("GPU W-проекция: CUDA / CPU проверка совпадения пройдена.")
else:
    print("GPU нет: проекция W на CPU (в конфигурации REQUIRE_GPU=False)")

In [ ]:
# 7. Подготовка данных на /content: никаких расчётов поверх Drive FUSE
import json
STAGED_INPUT = WORK_ROOT / "inputs" / RUN_ID
STAGED_INPUT.mkdir(parents=True, exist_ok=True)
if RUN_PROFILE == "SMOKE":
    from semmap_haken.hypothesis.synthetic_benchmarks import planted_typed_graph
    synthetic = planted_typed_graph(seed=SEED, nested=True, n_groups=4, per_group=20)
    g = synthetic.graph
    INPUT_LOCAL = STAGED_INPUT / "synthetic_truth_h1.tsv"
    with INPUT_LOCAL.open("w", encoding="utf-8") as fd:
        for item, u,v,r,w in zip(g.assertion_ids,g.source,g.target,g.relation,g.weight):
            fd.write(f"{item}\t{g.relation_names[int(r)]}\t{g.node_ids[int(u)]}\t"
                     f"{g.node_ids[int(v)]}\t{json.dumps({'weight': float(w)})}\n")
else:
    if not CONCEPTNET_TSV_ON_DRIVE:
        raise ValueError("Для PILOT/FULL укажите CONCEPTNET_TSV_ON_DRIVE в первой ячейке")
    source = Path(CONCEPTNET_TSV_ON_DRIVE)
    if not source.is_file():
        raise FileNotFoundError(f"Не найден исходный TSV: {source}")
    if source.name.endswith(".jsonl"):
        raise ValueError("Нужен исходный 5-колоночный TSV, не агрегированный selected_edges.jsonl")
    INPUT_LOCAL = STAGED_INPUT / source.name
    if not INPUT_LOCAL.exists() or INPUT_LOCAL.stat().st_size != source.stat().st_size:
        shutil.copy2(source, INPUT_LOCAL)

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_SHA = file_hash(INPUT_LOCAL)
print({"input": str(INPUT_LOCAL), "bytes": INPUT_LOCAL.stat().st_size, "sha256": DATA_SHA,
       "mode": RUN_PROFILE, "warning": "SYNTHETIC ONLY" if RUN_PROFILE == "SMOKE" else None})

## 8. Контрольные точки на Google Drive

Скрипт H1 сохраняет завершённые оптимизации парами JSON/NPZ. Но Colab может удалить виртуальную машину. Поэтому ноутбук регулярно копирует неизменяемые готовые результаты в **отдельные подпапки по контрольному графу** (без >10 тысяч файлов в одной папке) и восстанавливает их при следующем запуске.

Промежуточные массивы графов (data/) не копируются в Drive: при RESUME они пересоздаются из зафиксированного входа и случайных состояний. Маркер COMPLETED копируется последним. Локальная синхронизация через Google Drive может не быть мгновенной; последняя часть расчёта после непредвиденного обрыва может потребовать повторения.

In [ ]:
# 8. Минимальный менеджер снимков: изменённые файлы -> Drive, не все графы.
def checkpoint_group(task_name):
    # real-* | null_configuration_0000-* | null_conditional_flat_dc_sbm_0000-*
    return task_name.split("-", 1)[0]

def copy_if_new(src, dst):
    if not src.is_file():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return False  # записи задач неизменяемы, каждый RUN_ID уникален
    partial = dst.with_name(dst.name + ".part")
    shutil.copy2(src, partial)
    os.replace(partial, dst)
    return True

def restore_run(local_run, drive_run):
    """Restore checkpoint shards to the flat tasks directory expected by runner."""
    if not USE_DRIVE or not (drive_run / "manifest.json").is_file():
        return False
    if local_run.exists() and any(local_run.iterdir()):
        raise RuntimeError(f"Локальный запуск уже содержит данные: {local_run}")
    copy_if_new(drive_run / "manifest.json", local_run / "manifest.json")
    for root in ("nulls", "gpu_projection"):
        directory = drive_run / root
        if directory.exists():
            for src in directory.rglob("*"):
                if src.is_file() and not src.name.endswith(".part"):
                    copy_if_new(src, local_run / root / src.relative_to(directory))
    checkpoint_dir = drive_run / "checkpoints"
    if checkpoint_dir.exists():
        for src in checkpoint_dir.rglob("*"):
            if src.is_file() and src.suffix in {".json", ".npz"}:
                copy_if_new(src, local_run / "tasks" / src.name)
    for name in ("report.json", "COMPLETED"):
        copy_if_new(drive_run / name, local_run / name)
    return True

def persist_run(local_run, drive_run):
    """Sharded Drive snapshots; never persist ephemeral graph data/ subfolders."""
    if not USE_DRIVE or not local_run.exists():
        return 0
    copies = copy_if_new(local_run / "manifest.json", drive_run / "manifest.json")
    for root in ("tasks", "nulls", "gpu_projection"):
        source_dir = local_run / root
        if not source_dir.exists():
            continue
        for src in source_dir.rglob("*"):
            if not src.is_file() or src.name.endswith((".part", ".tmp")):
                continue
            rel = src.relative_to(source_dir)
            target = (drive_run / "checkpoints" / checkpoint_group(src.stem) / src.name
                      if root == "tasks" else drive_run / root / rel)
            copies += copy_if_new(src, target)
    copies += copy_if_new(local_run / "report.json", drive_run / "report.json")
    copies += copy_if_new(local_run / "COMPLETED", drive_run / "COMPLETED")
    return copies

def restore_checkpoints():
    return restore_run(RUN_LOCAL, RUN_DRIVE)

def persist_checkpoints():
    return persist_run(RUN_LOCAL, RUN_DRIVE)

restored = restore_checkpoints() if USE_DRIVE and not RUN_LOCAL.exists() else False
MODE = "RESUME" if (RUN_LOCAL / "manifest.json").exists() else "NEW"
print({"restored_from_drive": restored, "mode": MODE, "run_id": RUN_ID})

In [ ]:
# 9. Формируем CPU-задачу. taskset ограничивает весь процесс и дочерние worker'ы <= 12 ядрами.
params = PROFILES[RUN_PROFILE]
RESTARTS, NULLS = params["restarts"], params["nulls"]
TOTAL_FITS = 3 * RESTARTS * (1 + 2 * NULLS)
print(f"Модельных оптимизаций: {TOTAL_FITS}. Нулевые ансамбли: {2*NULLS}.")
if RUN_PROFILE == "FULL":
    print("ПРЕДУПРЕЖДЕНИЕ: 5970 подгонок может превышать ресурс одного сеанса Colab; необходимы контрольные точки.")
if RUN_PROFILE == "PILOT":
    print("PILOT предназначен для диагностики; 9 реализаций ансамбля недостаточно для серьёзного p-вывода.")

H1_ARGS = [str(ENV_PY), "-m", "semmap_haken.hypothesis.h1_cli",
           "--input", str(INPUT_LOCAL), "--output", str(RUN_LOCAL),
           "--mode", MODE, "--seed", str(SEED), "--restarts", str(RESTARTS),
           "--nulls", str(NULLS), "--workers", "auto",
           "--negatives", str(params["negatives"]), "--bootstrap", "500"]

CPU_IDS = sorted(os.sched_getaffinity(0))[:ALLOTTED_CPU] if hasattr(os, "sched_getaffinity") else []
if CPU_IDS and shutil.which("taskset"):
    H1_COMMAND = ["taskset", "-c", ",".join(map(str, CPU_IDS))] + H1_ARGS
else:
    if ALLOTTED_CPU < AVAILABLE_CPU:
        raise RuntimeError("Нужна команда taskset для строгого бюджета 12 CPU; установите util-linux")
    H1_COMMAND = H1_ARGS
    print("taskset отсутствует, но все выделенные ядра входят в CPU-бюджет")
print("Проверка: Python graph-tool =", ENV_PY)
print("CPU-бюджет:", ALLOTTED_CPU, "распределяется между OpenMP и процессами без oversubscription.")

In [ ]:
# 10. Запуск: периодическое резервирование задач в Drive, корректное RESUME
LOGS = WORK_ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
MAIN_LOG = LOGS / f"{RUN_ID}.log"

def last_log_lines(path, n=4):
    if not path.exists():
        return []
    with path.open("rb") as f:
        f.seek(max(0, f.seek(0, 2) - 5000))
        return f.read().decode("utf-8", errors="replace").splitlines()[-n:]

if (RUN_LOCAL / "COMPLETED").exists() and (RUN_LOCAL / "report.json").exists():
    print("Этот RUN_ID уже завершён: повторное обучение не запускается.")
else:
    with MAIN_LOG.open("a", encoding="utf-8") as log:
        log.write(f"\n=== START {time.strftime('%Y-%m-%d %H:%M:%S')} {MODE} ===\n")
        log.flush()
        process = subprocess.Popen(H1_COMMAND, cwd=REPOSITORY, env=CPU_ENV,
                                   stdout=log, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(CHECKPOINT_INTERVAL_SECONDS)
                n_synced = persist_checkpoints()
                done = len(list((RUN_LOCAL / "tasks").glob("*.json"))) if (RUN_LOCAL / "tasks").exists() else 0
                print(f"Готово модельных задач: {done}/{TOTAL_FITS}; обновлено файлов на Drive: {n_synced}")
            return_code = process.wait()
        except KeyboardInterrupt:
            print("Прерывание: останавливаем процесс, сохраняем готовые контрольные точки.")
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            persist_checkpoints()
            raise
        finally:
            n_synced = persist_checkpoints()
    print("Завершение процесса:", return_code, "| файлов синхронизировано:", n_synced)
    if return_code:
        print("\n".join(last_log_lines(MAIN_LOG, 25)))
        raise RuntimeError("H1 завершился с ошибкой; исправьте причину и повторите RUN_ID с RESUME")
assert (RUN_LOCAL / "report.json").is_file(), "Нет отчёта H1"
assert (RUN_LOCAL / "COMPLETED").is_file(), "Нет маркера COMPLETE: эксперимент не завершён"
persist_checkpoints()

In [ ]:
# 11. Отчёт: H1 не считается доказанной только по низкому MDL
REPORT = json.loads((RUN_LOCAL / "report.json").read_text(encoding="utf-8"))
print("СТАТУС:", REPORT["status"], "|", REPORT["decision"].get("reason"))
print("Исходная сеть / train / validation / test:", REPORT["counts"])
print("Выигрыш nSBM относительно DC-SBM по MDL (биты):",
      REPORT["observed_mdl_gain_bits"])
for model, info in REPORT["fits"].items():
    print(model, "| best MDL:", info["best_mdl_bits"],
          "| блоков по уровням:", info["blocks_by_level"],
          "| диапазон MDL:", info["range_mdl_bits"])
for ensemble, result in REPORT["nulls"].items():
    print("Нулевой ансамбль:", ensemble, "значимость:", result["significance"])
for model, result in REPORT["prediction"].items():
    print("Ранжирование теста", model, "MRR:",
          result["test"]["mrr"], "Hits@10:", result["test"]["hits_at_10"])
if NULLS < 99:
    print("ВНИМАНИЕ: недостаточно нулевых реализаций для надёжного заключения H1.")
print("Это условный результат на данном графе/выборке; научную калибровку выполняют отдельно.")

In [ ]:
# 11а. Визуальное сравнение MDL-выигрыша с распределениями нулевых контролей
import matplotlib.pyplot as plt
null_arrays = {
    name: details["mdlgain_controls"]
    for name, details in REPORT["nulls"].items()
    if details["mdlgain_controls"]
}
if null_arrays:
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, array in null_arrays.items():
        ax.hist(array, bins=min(18, max(4, len(array) // 3)),
                alpha=0.55, label=name)
    ax.axvline(REPORT["observed_mdl_gain_bits"], linewidth=2.0,
               linestyle="--", label="Исходный граф")
    ax.set(xlabel="Выигрыш вложенной nSBM над DC-SBM (биты)",
           ylabel="Число контрольных графов",
           title="Сравнение реальной структуры с нулевыми моделями")
    ax.legend()
    plt.show()
else:
    print("Нет нулевых реализаций: для SMOKE-графа распределение не строится.")

In [ ]:
# 12. CUDA-проекция W по найденным вложенным блокам; точность сравниваем с CPU.
# Это постобработка графонной структуры и НЕ часть H1 MDL-прогона.
if GPU_VISIBLE:
    from semmap_haken.hypothesis.graph import TypedGraph
    from semmap_haken.hypothesis.projection import estimate_block_kernel
    train_folder = RUN_LOCAL / "data" / "train"
    meta = json.loads((train_folder / "ids.json").read_text(encoding="utf-8"))
    training = TypedGraph(
        tuple(meta["node_ids"]), tuple(meta["relation_names"]),
        *(np.load(train_folder / f"{name}.npy", mmap_mode="r")
          for name in ("source", "target", "relation", "weight")),
        tuple(meta["assertion_ids"]))
    task_files = sorted((RUN_LOCAL / "tasks").glob("real-nested_DC_SBM-restart_*.json"))
    if not task_files:
        raise RuntimeError("Не найдено ни одной реальной nSBM-оптимизации")
    winning = min(task_files, key=lambda p: json.loads(p.read_text())["mdl_bits"])
    with np.load(winning.with_suffix(".npz"), allow_pickle=False) as saved:
        labels = [saved[f"p{i}"].copy() for i in range(len(saved.files))]
    # Независимая CPU / CUDA проверка на фиксированном подмножестве обучающих рёбер
    subset = training.edge_subset(np.arange(training.edge_count) < min(20_000, training.edge_count))
    cpu = estimate_block_kernel(subset, labels[0], device="cpu", batch_size=GPU_BATCH_SIZE)
    cuda = estimate_block_kernel(subset, labels[0], device="cuda", batch_size=GPU_BATCH_SIZE)
    def numeric_entries(result):
        return [(v["relation"],v["source_block"],v["target_block"],
                 v["edge_count"],v["weight_sum"]) for v in result["entries"]]
    a, b = numeric_entries(cpu), numeric_entries(cuda)
    assert len(a) == len(b)
    for v, w in zip(a, b):
        assert v[:4] == w[:4] and np.isclose(v[4], w[4], rtol=1e-9, atol=1e-7)
    print("Проверено совпадение CPU и CUDA на", subset.edge_count, "рёбрах.")
    outputs = RUN_LOCAL / "gpu_projection"
    outputs.mkdir(parents=True, exist_ok=True)
    for i, level in enumerate(labels[:GPU_MAX_LEVELS]):
        if len(set(level.tolist())) <= 1:
            continue
        projected = estimate_block_kernel(training, level,
                                          device="cuda", batch_size=GPU_BATCH_SIZE)
        assert projected["backend"] == "cuda"
        out = outputs / f"empirical_W_level_{i:03d}.json"
        temp = out.with_suffix(".tmp")
        temp.write_text(json.dumps(projected, ensure_ascii=False), encoding="utf-8")
        os.replace(temp, out)
        print("GPU:", out.name, "блоков:", len(projected["block_mass"]),
              "ненулевых пар:", len(projected["entries"]))
    persist_checkpoints()
else:
    print("GPU недоступен, CUDA-проекция пропущена; H1 nSBM остаётся CPU экспериментом.")

## 13. Итог и следующий научный шаг

Содержимое **report.json** — результаты вычислительного теста H1 при выбранном графе и двух определённых нулевых ансамблях. Отдельный файлик **gpu_projection/empirical_W_level_*.json** — эмпирический W-only графон, а **не** полный графекс (S/I отсутствуют).

Для научного заключения необходимо выполнить калибровку на **сериях независимых плоских и иерархических искусственных графов** с тем же числом повторов, затем изучить ConceptNet-10k и ConceptNet-100k, а в следующих экспериментах — устойчивость при возмущениях и независимые источники. Штатный FULL-профиль не гарантирует статистической мощности без предварительной калибровки.

Возобновление после обрыва Colab: откройте этот ноутбук в новом сеансе, оставьте **тот же RUN_ID, SOURCE_REVISION и параметры**, выполните ячейки сверху вниз. Снимки будут автоматически восстановлены с Google Drive.

In [ ]:
# 13. Необязательная серия научной калибровки (дорогая; отдельно от H1 на ConceptNet)
if RUN_CALIBRATION:
    CAL_LOCAL = WORK_ROOT / "calibration" / f"{RUN_ID}-calibration"
    CAL_DRIVE = DRIVE_ROOT / "colab" / "hypothesis" / "calibration" / f"{RUN_ID}-calibration"
    if USE_DRIVE:
        CAL_DRIVE.mkdir(parents=True, exist_ok=True)
    calc = [str(ENV_PY), str(REPOSITORY / "scripts" / "calibrate_h1_synthetic.py"),
            "--output", str(CAL_LOCAL), "--simulations", str(CALIBRATION_SIMULATIONS),
            "--restarts", str(CALIBRATION_RESTARTS), "--nulls", str(CALIBRATION_NULLS),
            "--workers", "auto", "--resume"]
    # Restore each artificial graph's own checkpoint. No flat >10k-file Drive folders.
    if USE_DRIVE:
        for condition in ("flat", "nested"):
            for i in range(CALIBRATION_SIMULATIONS):
                local_one = CAL_LOCAL / condition / f"run-{i:03d}"
                drive_one = CAL_DRIVE / condition / f"run-{i:03d}"
                if (drive_one / "manifest.json").exists() and not local_one.exists():
                    restore_run(local_one, drive_one)

    def persist_calibration():
        copied = 0
        for condition in ("flat", "nested"):
            for i in range(CALIBRATION_SIMULATIONS):
                local_one = CAL_LOCAL / condition / f"run-{i:03d}"
                drive_one = CAL_DRIVE / condition / f"run-{i:03d}"
                copied += persist_run(local_one, drive_one)
        copied += copy_if_new(CAL_LOCAL / "calibration.json",
                              CAL_DRIVE / "calibration.json")
        return copied

    run_log = LOGS / f"{RUN_ID}-calibration.log"
    print("Это отдельный эксперимент на независимых искусственных графах.")
    print("Оптимизаций:", 2 * CALIBRATION_SIMULATIONS *
          3 * CALIBRATION_RESTARTS * (1 + 2 * CALIBRATION_NULLS))
    with run_log.open("a") as out:
        proc = subprocess.Popen(
            ["taskset", "-c", ",".join(map(str, CPU_IDS))] + calc
            if CPU_IDS and shutil.which("taskset") else calc,
            cwd=REPOSITORY, env=CPU_ENV, stdout=out, stderr=subprocess.STDOUT)
        try:
            while proc.poll() is None:
                time.sleep(CHECKPOINT_INTERVAL_SECONDS)
                print("Калибровка: новых файлов на Drive:", persist_calibration())
            rc = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            try:
                proc.wait(timeout=20)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            persist_calibration()
            raise
        finally:
            persist_calibration()
    if rc != 0:
        raise RuntimeError("Калибровка прервалась: см. журнал; сохраните RUN_ID и повторите")
    print("Калибровка:", CAL_DRIVE / "calibration.json")
else:
    print("Калибровка не запускалась (RUN_CALIBRATION=False); итог H1 — предварительный.")